<a href="https://colab.research.google.com/github/elbuh0/genai-course/blob/main/genai-course/week3/Week_3_Lab3_LLM_APIs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 3 — Multi-Provider LLM APIs
**COM SCI-X 455 · Week 3 · Megan Armani, PhD**

> Push to `genai-course/week3/`

---

## Setup

In [2]:
!pip install openai anthropic rouge-score pydantic huggingface_hub --quiet

import os
# Store keys in Colab Secrets (left sidebar lock icon) — never hardcode
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('openai')
os.environ['ANTHROPIC_API_KEY'] = userdata.get('anthropic')

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 33.3 MB/s eta 0:00:00


## Part 1 — Basic API Calls: OpenAI + Anthropic + Llama 3

In [3]:
from openai import OpenAI
import anthropic

openai_client    = OpenAI()
anthropic_client = anthropic.Anthropic()

ARTICLE = """
A Roadside Picnic is a philosophical science fiction novel by the Soviet authors Arkady and Boris Strugatsky. It was written in 1971 and published in 1972, and is their most popular and most widely translated novel outside the former Soviet Union. As of 2003, Boris Strugatsky counted 55 publications of Roadside Picnic in 22 countries.

It is set in the aftermath of an extraterrestrial event, the Visitation, which occurred simultaneously in several locations around the Earth over a two-day period. Neither the Visitors themselves nor their means of arrival or departure were ever seen by the local populations, who lived inside the relatively small areas, each a few square kilometers, of the six Visitation Zones. The Zones exhibit strange and dangerous phenomena that are not understood by humans and contain artifacts with inexplicable properties. The title of the novel derives from an analogy that is proposed by the character Dr. Valentine Pilman, who compares the Visitation to a picnic:
A picnic. Picture a forest, a country road, a meadow. Cars drive off the country road into the meadow, a group of young people get out carrying bottles, baskets of food, transistor radios, and cameras. They light fires, pitch tents, turn on the music. In the morning they leave. The animals, birds, and insects that watched in horror through the long night creep out from their hiding places. And what do they see? Old spark plugs and old filters strewn around... Rags, burnt-out bulbs, and a monkey wrench left behind... And of course, the usual mess—apple cores, candy wrappers, charred remains of the campfire, cans, bottles, somebody’s handkerchief, somebody’s penknife, torn newspapers, coins, faded flowers picked in another meadow.

In this analogy, the nervous animals are the humans who venture forth after the Visitors have left, discovering items and anomalies that are ordinary to those who have discarded them, but incomprehensible or deadly to the earthlings. The explanation implies that the Visitors may not have paid any attention to or even noticed Earth's inhabitants during their visit, just as many humans do not notice or pay attention to insects and wildlife during a picnic. The artifacts and phenomena that are left behind by the Visitors in the Zones were garbage, which are discarded and forgotten without any intentions to advance or damage humanity. There is little chance that the Visitors will return again because for them it was a brief stop, for reasons unknown, on the way to their actual destination.
The novel is set in a post-Visitation world in which there are now six zones that are known on Earth to be full of unexplained phenomena with strange happenings having briefly occurred. They are assumed to have been Visitations by aliens. Governments and the United Nations, fearful of unforeseen consequences, try to keep tight control over them to prevent leakage of artifacts from the Zones. A subculture of stalkers, scavengers who go into the zones to steal the artifacts for profit, has evolved around the zones. The novel is set in and around a specific zone in Harmont, a fictitious town in an unspecified English-speaking country, and follows the protagonist over the course of eight years.
As a vignette, in Section 4, Redrick goes into the Zone one last time to reach the wish-granting "Golden Sphere". He has a map that was given to him by Burbridge, whose son, Arthur, joins him on the expedition. Redrick knows one of them has to die in order to temporarily deactivate an invisible phenomenon known as the "meatgrinder" for the other to reach the Sphere, but he keeps that a secret from Arthur, whom he intends to sacrifice. After they get to the location, having survived many obstacles, Arthur rushes towards the Sphere and shouts out selfless wishes for a better world, only to be savagely dispatched by the meatgrinder. With the Sphere in front of him, an exhausted Redrick looks back in confusion and bitterness on his whole life of struggle in a harsh world, and finds that he cannot articulate what he actually wants from the Sphere. He hopes the Sphere will look into his soul and "figure out" his wish because "it can't be bad." In the end he finds nothing better than to repeat the dead youth's words, "HAPPINESS FOR EVERYBODY, FREE OF CHARGE, AND MAY NO ONE BE LEFT BEHIND!"

"""

In [4]:
# Call GPT-4o to summarise ARTICLE in exactly 2 sentences

import time

start = time.perf_counter()

response = openai_client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {
            "role": "user",
            "content": f"Summarise the following article in exactly 2 sentences:\n\n{ARTICLE}"
        }
    ]
)

gpt_summary = response.choices[0].message.content
gpt_latency = time.perf_counter() - start

print(gpt_summary)


A Roadside Picnic by Arkady and Boris Strugatsky explores a post-extraterrestrial event world where mysterious Zones left by alien Visitations are filled with dangerous phenomena and incomprehensible artifacts, leading to the rise of scavenger stalkers who risk their lives for profit. The novel follows Redrick, a stalker who, while seeking a wish-granting artifact, faces moral dilemmas and ultimately desires universal happiness, reflecting on the chaotic nature of human desires and existence.


In [5]:
# Claude summary

import time

start = time.perf_counter()

claude_response = anthropic_client.messages.create(
    model="claude-fable-5-1",
    max_tokens=300,
    system="Summarise the article in exactly 2 sentences.",
    messages=[
        {
            "role": "user",
            "content": ARTICLE
        }
    ]
)

claude_summary = claude_response.content[0].text
claude_latency = time.perf_counter() - start

print(claude_summary)



Roadside Picnic, a philosophical science fiction novel written in 1971 by Soviet authors Arkady and Boris Strugatsky, is set after an alien Visitation left six Zones on Earth filled with dangerous phenomena and inexplicable artifacts, which scavengers called stalkers risk their lives to retrieve for profit despite government efforts to contain them. The title comes from Dr. Valentine Pilman's analogy that the aliens merely stopped briefly and left behind their garbage like picnickers indifferent to the insects around them, and the story follows the stalker Redrick over eight years, culminating in his sacrifice of young Arthur to reach the wish-granting Golden Sphere, where, unable to articulate his own desire, he repeats the dead youth's plea: "Happiness for everybody, free of charge, and may no one be left behind!"


In [6]:
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("huggingface")


In [7]:
# TODO: Call Llama 3.1 8B Instruct via Hugging Face.
# Compare GPT-4o, Claude Fable 5.1, and Llama 3.1 8B Instruct.

from huggingface_hub import InferenceClient
import time
import re
import pandas as pd
from IPython.display import display


# ---------- Call Llama 3.1 8B Instruct ----------

hf_client = InferenceClient(
    provider="featherless-ai",
    api_key=os.environ["HF_TOKEN"]
)

start = time.perf_counter()

llama_response = hf_client.chat_completion(
    model="meta-llama/Llama-3.1-8B-Instruct",
    messages=[
        {
            "role": "user",
            "content": f"""
Summarise the following article.
Use no more than 3 paragraphs.
Put each sentence on its own line.

ARTICLE:

{ARTICLE}
"""
        }
    ],
    max_tokens=500
)

llama_summary = llama_response.choices[0].message.content
llama_latency = time.perf_counter() - start


# ---------- Helper functions ----------

def word_count(text):
    return len(re.findall(r"\b[\w'-]+\b", text))


def format_summary(text):
    """
    Display the summary using at most 3 paragraphs.
    Preserve the model's paragraph structure where possible.
    """
    paragraphs = re.split(r"\n\s*\n", text.strip())
    paragraphs = [p.strip() for p in paragraphs if p.strip()]

    if len(paragraphs) <= 3:
        return "\n\n".join(paragraphs)

    # Keep the first two paragraphs and combine the remainder
    return "\n\n".join([
        paragraphs[0],
        paragraphs[1],
        " ".join(paragraphs[2:])
    ])


# ---------- Generic factual consistency check ----------

def factual_consistency(summary):
    """
    Ask GPT-4o to assess whether the summary is supported by ARTICLE.

    This is a general-purpose heuristic rather than a definitive
    hallucination detector.
    """

    check = openai_client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are evaluating factual consistency. "
                    "Determine whether the summary contains claims "
                    "that are unsupported or contradicted by the source article. "
                    "Respond with only one of: "
                    "Likely supported, Review."
                )
            },
            {
                "role": "user",
                "content": f"""
SOURCE ARTICLE:
{ARTICLE}

SUMMARY:
{summary}
"""
            }
        ],
        max_tokens=10
    )

    return check.choices[0].message.content.strip()


# ---------- Format summaries ----------

gpt_formatted = format_summary(gpt_summary)
claude_formatted = format_summary(claude_summary)
llama_formatted = format_summary(llama_summary)


# ---------- Print summaries ----------

print("=== GPT-4o ===")
print(gpt_formatted)

print("\n=== Claude Fable 5.1 ===")
print(claude_formatted)

print("\n=== Llama 3.1 8B Instruct ===")
print(llama_formatted)


# ---------- Comparison table ----------

comparison = pd.DataFrame([
    {
        "Model": "GPT-4o",
        "Word Count": word_count(gpt_summary),
        "Latency (s)": round(gpt_latency, 3),
        "Factual Consistency": factual_consistency(gpt_summary)
    },
    {
        "Model": "Claude Fable 5.1",
        "Word Count": word_count(claude_summary),
        "Latency (s)": round(claude_latency, 3),
        "Factual Consistency": factual_consistency(claude_summary)
    },
    {
        "Model": "Llama 3.1 8B Instruct",
        "Word Count": word_count(llama_summary),
        "Latency (s)": round(llama_latency, 3),
        "Factual Consistency": factual_consistency(llama_summary)
    }
])

print("\n=== Model Comparison ===")
display(comparison)


=== GPT-4o ===
A Roadside Picnic by Arkady and Boris Strugatsky explores a post-extraterrestrial event world where mysterious Zones left by alien Visitations are filled with dangerous phenomena and incomprehensible artifacts, leading to the rise of scavenger stalkers who risk their lives for profit. The novel follows Redrick, a stalker who, while seeking a wish-granting artifact, faces moral dilemmas and ultimately desires universal happiness, reflecting on the chaotic nature of human desires and existence.

=== Claude Fable 5.1 ===
Roadside Picnic, a philosophical science fiction novel written in 1971 by Soviet authors Arkady and Boris Strugatsky, is set after an alien Visitation left six Zones on Earth filled with dangerous phenomena and inexplicable artifacts, which scavengers called stalkers risk their lives to retrieve for profit despite government efforts to contain them. The title comes from Dr. Valentine Pilman's analogy that the aliens merely stopped briefly and left behind th

,Model,Word Count,Latency (s),Factual Consistency
0,GPT-4o,71,3.572,Likely supported
1,Claude Fable 5.1,128,8.077,Likely supported
2,Llama 3.1 8B Instruct,67,7.175,Likely supported


Quality rating: GPT-4o: 3; Claude Fable: 1; Llama: 2. I could not tell which is which based on review of output alone, though GPT-4o and Llama 3.1 appeared consistently more concise. All were regullarly factually consistent, i.e., did not hallucinate.

## Part 2 — Streaming

In [16]:
# TODO: Implement streaming for GPT-4o:
# - Use stream=True
# - Print tokens as they arrive
# - Measure time to first token vs total time
# - Compare streaming to a non-streaming call

import time
import json
from pydantic import BaseModel, field_validator
from typing import Literal

# 1. GPT-4o STREAMING

print("=== Streaming GPT-4o ===\n")

start_time = time.perf_counter()
first_token_time = None
full_text = ""

stream = openai_client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {
            "role": "user",
            "content": "Explain the Golden Sphere in Roadside Picnic."
        }
    ],
    stream=True
)

for chunk in stream:
    delta = chunk.choices[0].delta.content or ""

    if delta:
        if first_token_time is None:
            first_token_time = time.perf_counter()

        print(delta, end="", flush=True)
        full_text += delta

end_time = time.perf_counter()

time_to_first_token = first_token_time - start_time
total_streaming_time = end_time - start_time

print("\n\n--- Streaming Metrics ---")
print(f"Time to first token: {time_to_first_token:.3f} seconds")
print(f"Total streaming time: {total_streaming_time:.3f} seconds")
print(f"Total characters: {len(full_text)}")

# 2. GPT-4o NON-STREAMING

print("\n=== Non-Streaming GPT-4o ===\n")

start_time = time.perf_counter()

response = openai_client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {
            "role": "user",
            "content": "Explain the Golden Sphere in Roadside Picnic."
        }
    ]
)

non_streaming_text = response.choices[0].message.content

end_time = time.perf_counter()

non_streaming_time = end_time - start_time

print(non_streaming_text)

print("\n--- Non-Streaming Metrics ---")
print(f"Total response time: {non_streaming_time:.3f} seconds")


# 3. STREAMING VS NON-STREAMING COMPARISON

print("\n=== Streaming Comparison ===")

streaming_comparison = {
    "Streaming time to first token (s)": round(time_to_first_token, 3),
    "Streaming total time (s)": round(total_streaming_time, 3),
    "Non-streaming total time (s)": round(non_streaming_time, 3)
}

for metric, value in streaming_comparison.items():
    print(f"{metric}: {value}")


=== Streaming GPT-4o ===

In "Roadside Picnic," a science fiction novel by Arkady and Boris Strugatsky, the Golden Sphere is an enigmatic artifact located within the Zone, a heavily contaminated and hazardous area filled with mysterious and potentially dangerous items left behind by an extraterrestrial visitation. The Zone is a focal point of the novel, attracting scientists, government agents, and "stalkers," who are illegal scavengers risking their lives to retrieve alien artifacts for profit.

The Golden Sphere, also known as the Wish Machine, is one of the most legendary and sought-after artifacts in the Zone due to its rumored ability to grant any wish. This powerful and ambiguous artifact symbolizes humanity's deepest desires and the inherent risks and moral dilemmas of attaining them. Its powers are not well understood, and the consequences of wishes granted by the Sphere can be unpredictable, embodying the novel's themes of unintended consequences and the limitations of human u

## Part 3 — Structured Output with Pydantic

In [9]:

# 4. STRUCTURED OUTPUT WITH PYDANTIC

class ArticleAnalysis(BaseModel):
    title: str
    summary: str
    sentiment: Literal["positive", "negative", "neutral", "mixed"]
    confidence: float
    key_claims: list[str]

    @field_validator("summary")
    @classmethod
    def summary_must_have_two_sentences(cls, v):
        sentences = [
            s for s in v.replace("!", ".").replace("?", ".").split(".")
            if s.strip()
        ]

        if len(sentences) != 2:
            raise ValueError(
                f"summary must contain exactly 2 sentences; got {len(sentences)}"
            )

        return v

    @field_validator("confidence")
    @classmethod
    def confidence_must_be_valid(cls, v):
        if not 0.0 <= v <= 1.0:
            raise ValueError("confidence must be between 0.0 and 1.0")

        return v

    @field_validator("key_claims")
    @classmethod
    def must_have_three_claims(cls, v):
        if len(v) != 3:
            raise ValueError(
                f"must have 3 key claims; got {len(v)}"
            )

        return v

def analyze_article(article_text: str) -> ArticleAnalysis:

    response = openai_client.chat.completions.create(
        model="gpt-4o",
        response_format={"type": "json_object"},
        messages=[
            {
                "role": "system",
                "content": (
                    "Analyze the article and return ONLY valid JSON. "
                    "The JSON must follow the provided schema. "
                    "The summary must contain exactly 2 sentences. "
                    "Include exactly 3 key claims."
                )
            },
            {
                "role": "user",
                "content": (
                    f"Schema:\n"
                    f"{json.dumps(ArticleAnalysis.model_json_schema(), indent=2)}\n\n"
                    f"Article:\n{article_text}"
                )
            }
        ]
    )

    return ArticleAnalysis.model_validate_json(
        response.choices[0].message.content
    )


# 5. RUN STRUCTURED ANALYSIS

print("\n=== Structured Article Analysis ===\n")

try:
    analysis = analyze_article(ARTICLE)

    print(analysis.model_dump_json(indent=2))

except Exception as e:
    print(f"Validation failed: {e}")


=== Structured Article Analysis ===

{
  "title": "A Roadside Picnic: A Philosophical Exploration of Post-Visitation Earth",
  "summary": "The novel 'A Roadside Picnic', by Arkady and Boris Strugatsky, explores a world transformed by an extraterrestrial event known as the Visitation, leaving behind mysterious and dangerous artifacts in several zones around the world. The story follows the protagonist Redrick as he navigates these perilous zones, ultimately seeking a wish-granting device for happiness amid moral and existential dilemmas.",
  "sentiment": "neutral",
  "confidence": 0.95,
  "key_claims": [
    "The Visitation event left Earth with enigmatic artifacts, altering human understanding of technology and existence.",
    "Stalkers risk their lives extracting these artifacts from the zones, driven by greed and survival instincts.",
    "Despite global efforts to control the zones, the artifacts' potential and the possibility of Visitors returning remain ambiguous."
  ]
}


In [10]:
# TODO: Call GPT-4o with response_format={'type':'json_object'} to analyse ARTICLE.
# Ask it to return JSON matching ArticleSummary schema.
# Parse with ArticleSummary.model_validate_json(response).
# Show what happens when confidence=1.5 — does your validator catch it?

from pydantic import BaseModel, field_validator
from typing import Literal


# ---------- Define schema ----------

class ArticleSummary(BaseModel):
    title: str
    summary: str
    sentiment: Literal["positive", "negative", "neutral", "mixed"]
    confidence: float
    key_claims: list[str]

    @field_validator("confidence")
    @classmethod
    def confidence_must_be_valid(cls, value):
        if not 0.0 <= value <= 1.0:
            raise ValueError("confidence must be between 0.0 and 1.0")
        return value

    @field_validator("key_claims")
    @classmethod
    def must_have_three_claims(cls, value):
        if len(value) != 3:
            raise ValueError(
                f"must have exactly 3 key claims; got {len(value)}"
            )
        return value


# ---------- Ask GPT-4o for structured JSON ----------

response = openai_client.chat.completions.create(
    model="gpt-4o",
    response_format={"type": "json_object"},
    messages=[
        {
            "role": "system",
            "content": (
                "Analyse the article and return ONLY valid JSON matching "
                "the ArticleSummary schema. Include exactly 3 key claims."
            )
        },
        {
            "role": "user",
            "content": ARTICLE
        }
    ]
)


# ---------- Parse and validate GPT-4o response ----------

try:
    analysis = ArticleSummary.model_validate_json(
        response.choices[0].message.content
    )

    print("=== Valid GPT-4o Response ===")
    print(analysis.model_dump_json(indent=2))

except Exception as e:
    print("Validation failed:")
    print(e)


# ---------- Test invalid confidence ----------

print("\n=== Testing confidence=1.5 ===")

try:
    invalid_analysis = ArticleSummary(
        title="Test Article",
        summary="This is the first sentence. This is the second sentence.",
        sentiment="neutral",
        confidence=1.5,
        key_claims=[
            "First claim",
            "Second claim",
            "Third claim"
        ]
    )

    print("Validation unexpectedly passed:")
    print(invalid_analysis)

except Exception as e:
    print("Validator caught the invalid confidence value:")
    print(e)



Validation failed:
6 validation errors for ArticleSummary
summary
  Field required [type=missing, input_value={'title': 'Roadside Picni...is life's struggles."}]}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
sentiment
  Field required [type=missing, input_value={'title': 'Roadside Picni...is life's struggles."}]}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
confidence
  Field required [type=missing, input_value={'title': 'Roadside Picni...is life's struggles."}]}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
key_claims.0
  Input should be a valid string [type=string_type, input_value={'claim': 'The novel depi...explicable properties.'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type
key_claims.1
  Input should be a valid string [type=string_type, input_value={'claim': "The Visitors' ...ter

## Part 4 — Evaluation with ROUGE

In [11]:
from rouge_score import rouge_scorer
scorer = rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)

REFERENCE = "Roadside Picnic by Arkady and Boris Strugatsky is a philosophical science fiction novel about the aftermath of an extraterrestrial Visitation that leaves six dangerous Zones filled with mysterious artifacts and phenomena. The story follows Redrick, a stalker who enters the Zone near Harmont to retrieve artifacts and eventually reaches the wish-granting Golden Sphere, where he struggles to express his deepest desire after his friend Arthur, a sacrificial lamb, is killed."


In [12]:
# TODO: Score all 3 model summaries (GPT-4o, Claude, Llama 3) against REFERENCE.
# Build a results table: Model | ROUGE-1 | ROUGE-2 | ROUGE-L
# Which model scored highest? Does the highest ROUGE score mean the best summary?

# TODO: Score all 3 model summaries against REFERENCE.
# Build a results table: Model | ROUGE-1 | ROUGE-2 | ROUGE-L
# Identify the highest-scoring model and discuss whether ROUGE
# necessarily means it is the best summary.

import pandas as pd


# ---------- Calculate ROUGE scores ----------

def calculate_rouge(summary):
    scores = scorer.score(REFERENCE, summary)

    return {
        "ROUGE-1": scores["rouge1"].fmeasure,
        "ROUGE-2": scores["rouge2"].fmeasure,
        "ROUGE-L": scores["rougeL"].fmeasure
    }


results = []

for model, summary in [
    ("GPT-4o", gpt_summary),
    ("Claude Fable 5.1", claude_summary),
    ("Llama 3.1 8B Instruct", llama_summary)
]:
    scores = calculate_rouge(summary)

    results.append({
        "Model": model,
        "ROUGE-1": round(scores["ROUGE-1"], 3),
        "ROUGE-2": round(scores["ROUGE-2"], 3),
        "ROUGE-L": round(scores["ROUGE-L"], 3)
    })


# ---------- Results table ----------

rouge_results = pd.DataFrame(results)

print("=== ROUGE Results ===")
display(rouge_results)


# ---------- Find highest-scoring model ----------

average_scores = rouge_results[
    ["ROUGE-1", "ROUGE-2", "ROUGE-L"]
].mean(axis=1)

highest_index = average_scores.idxmax()
highest_model = rouge_results.loc[highest_index, "Model"]
highest_score = average_scores.iloc[highest_index]

print(
    f"\nHighest average ROUGE score: "
    f"{highest_model} ({highest_score:.3f})"
)


# ---------- Interpretation ----------

print("\n=== Interpretation ===")
print(
    "A higher ROUGE score means the summary has greater word or "
    "phrase overlap with the reference summary."
)
print(
    "However, the highest ROUGE score does not necessarily mean "
    "the summary is the best. ROUGE rewards similarity to the "
    "reference wording and may not fully capture factual accuracy, "
    "clarity, completeness, or overall quality."
)


=== ROUGE Results ===


,Model,ROUGE-1,ROUGE-2,ROUGE-L
0,GPT-4o,0.500,0.169,0.319
1,Claude Fable 5.1,0.485,0.190,0.307
2,Llama 3.1 8B Instruct,0.457,0.130,0.243



Highest average ROUGE score: GPT-4o (0.329)

=== Interpretation ===
A higher ROUGE score means the summary has greater word or phrase overlap with the reference summary.
However, the highest ROUGE score does not necessarily mean the summary is the best. ROUGE rewards similarity to the reference wording and may not fully capture factual accuracy, clarity, completeness, or overall quality.


### ❓ Reflect
1. Did GPT-4o, Claude, and Llama 3 produce meaningfully different summaries? Describe the differences.
2. Your ROUGE scores are based on one reference. What's wrong with this evaluation methodology?
3. What would a better evaluation framework look like for summarisation quality?

1. I felt Claude was far more detailed and generally more accurate than either GPT-4o and Llama 3. The latter two generated text with similar level of detail and actually hallucination potential as discovered later in the lab. Perhaps this is due to Claude Fable being a bigger and better model.

2. The issue with basinc ROUGE scores on one reference, in this case an article summary, is that there is more than one way to write a good summary. In fact a better ROUGE score due to e.g. greater word overlap may be a worse summary as too verbose or repetitive.

3. A better evaluation methodology would have multiple elements, such as more than a single reference summary, LLM review as opposed to ROUGE-type scoring, and/or a definition of what a good summary would be, e.g. factual and concise for news, with an LLM as a judge ensuring statements are anchored in referenced facts, or more communicating of key themes for works of literature less sensitive to hallucination due to multliplicity of interpretations.

---
## Submission
- [ ] Push to `genai-course/week3/lab3.ipynb`
- [ ] All 3 models called and compared
- [ ] Pydantic validation demonstrated (including a failure case)
- [ ] ROUGE table completed

## Part 5 — Hallucination Detection
**Goal:** Identify hallucinated claims in model output. This feeds directly into your homework hallucination audit.

In [31]:
# TODO: Design 3 prompts likely to trigger hallucination.
# Run GPT-4o and Claude on each prompt.
# Manually verify claims as CORRECT / HALLUCINATED / UNVERIFIABLE.
# Build a results table.


import pandas as pd


# 1. TEST PROMPTS

prompts = [
    {
        "Prompt": (
            "In The Ghost Story of Tokaido Yotsuya, by Utagawa Kunisada "
            "AKA Utagawa Toyokuni III, Iemon, tormented by the ghost of Oiwa, "
            "has taken refuge in a hermitage at Hebiyama (lit. Snake Mountain). "
            "Emerging from a burning lantern, what does Oiwa's ghost do to "
            "Akiyama Chobe, Iemon's accomplice?"
        )
    },
    {
        "Prompt": (
            "Mexican composer Arturo Márquez's Máscaras, specifically "
            "La Pasión Según Marcos, opens with what four notes and by what instrument?"
        )
    },
    {
        "Prompt": (
            "In Polina Barskova's Living Pictures, specifically the chapter "
            "Estrangement, what are the walls in Berlin covered with?"
        )
    }
]


# 2. RUN GPT-4o AND CLAUDE

results = []

for test in prompts:

    prompt = test["Prompt"]


    # GPT-4o

    gpt_response = openai_client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    gpt_answer = gpt_response.choices[0].message.content

    # Claude

    claude_response = anthropic_client.messages.create(
        model="claude-fable-5-1",
        max_tokens=1000,
        thinking={
            "type": "adaptive"
        },
        output_config={
            "effort": "low"
        },
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    # Claude may return thinking blocks as well as text blocks.
    # Extract only the actual text response.
    claude_answer = "".join(
        block.text
        for block in claude_response.content
        if getattr(block, "type", None) == "text"
    )


    # 3. STORE RESULTS

    results.append({
        "Model": "GPT-4o",
        "Prompt": prompt,
        "Response": gpt_answer,
        "Verdict": ""
    })

    results.append({
        "Model": "Claude Fable 5.1",
        "Prompt": prompt,
        "Response": claude_answer,
        "Verdict": ""
    })

# 4. CREATE RESULTS TABLE

hallucination_results = pd.DataFrame(results)


# 5. DISPLAY RESULTS

print("=== Hallucination Test Results ===")

display(
    hallucination_results.style
    .set_properties(
        subset=["Model", "Prompt", "Response", "Verdict"],
        **{
            "direction": "ltr",
            "text-align": "left",
            "vertical-align": "top"
        }
    )
    .set_properties(
        subset=["Prompt"],
        **{
            "width": "350px",
            "white-space": "normal"
        }
    )
    .set_properties(
        subset=["Response"],
        **{
            "width": "700px",
            "white-space": "pre-wrap"
        }
    )
    .set_properties(
        subset=["Model", "Verdict"],
        **{
            "width": "150px"
        }
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "left"),
                ("direction", "ltr"),
                ("background-color", "#f2f2f2"),
                ("font-weight", "bold")
            ]
        },
        {
            "selector": "table",
            "props": [
                ("width", "100%"),
                ("table-layout", "auto"),
                ("direction", "ltr")
            ]
        }
    ])
)



=== Hallucination Test Results ===


,Model,Prompt,Response,Verdict
0,GPT-4o,"In The Ghost Story of Tokaido Yotsuya, by Utagawa Kunisada AKA Utagawa Toyokuni III, Iemon, tormented by the ghost of Oiwa, has taken refuge in a hermitage at Hebiyama (lit. Snake Mountain). Emerging from a burning lantern, what does Oiwa's ghost do to Akiyama Chobe, Iemon's accomplice?","In ""The Ghost Story of Tokaido Yotsuya"" by Utagawa Kunisada, also known as Utagawa Toyokuni III, Oiwa's ghost emerges from a burning lantern and terrifies Iemon's accomplice, Akiyama Chobe, causing him to die from panic and fright. Her vengeful spirit continues to haunt those involved in her murder, leading to their eventual downfall.",
1,Claude Fable 5.1,"In The Ghost Story of Tokaido Yotsuya, by Utagawa Kunisada AKA Utagawa Toyokuni III, Iemon, tormented by the ghost of Oiwa, has taken refuge in a hermitage at Hebiyama (lit. Snake Mountain). Emerging from a burning lantern, what does Oiwa's ghost do to Akiyama Chobe, Iemon's accomplice?","In this print (from the Hebiyama hermitage scene of *Tōkaidō Yotsuya Kaidan*), Oiwa's ghost emerges from the burning paper lantern with her long black hair streaming out and wraps it around Akiyama Chōbei, snaring him in her hair and strangling him as Iemon looks on in terror.",
2,GPT-4o,"Mexican composer Arturo Márquez's Máscaras, specifically La Pasión Según Marcos, opens with what four notes and by what instrument?","Arturo Márquez's piece ""Máscaras,"" specifically the movement ""La Pasión Según Marcos,"" opens with four prominent notes played by the piano. These opening notes set the tone for the rest of the movement.",
3,Claude Fable 5.1,"Mexican composer Arturo Márquez's Máscaras, specifically La Pasión Según Marcos, opens with what four notes and by what instrument?","I'm not able to confirm the exact pitches with confidence. What I can tell you is that *Máscaras* is Márquez's concerto for harp and orchestra, and its first movement, ""La Pasión según Marcos,"" opens with the solo harp alone stating a short, ostinato-like four-note motif that then seeds the rest of the movement's rhythmic drive. Various program notes describe this as a harp motif, but I don't have reliable confirmation of the specific four pitches, so I'd recommend checking the published score (Peermusic Classical) or the composer's own program note rather than trusting a guess from me.",
4,GPT-4o,"In Polina Barskova's Living Pictures, specifically the chapter Estrangement, what are the walls in Berlin covered with?","In Polina Barskova's ""Living Pictures,"" specifically in the chapter titled ""Estrangement,"" the walls in Berlin are described as being covered with posters, graffiti, and various layers of public artwork. These visual elements create a collage that reflects the city's diverse culture, history, and public expression.",
5,Claude Fable 5.1,"In Polina Barskova's Living Pictures, specifically the chapter Estrangement, what are the walls in Berlin covered with?","I'm not able to confirm this detail with confidence. Polina Barskova's *Living Pictures* (translated by Catherine Ciepiela) does include a piece set in Berlin, where the narrator moves through the city as an outsider, but I don't have a reliable recollection of the specific passage describing what the walls are covered with—whether it's graffiti, posters, bullet marks from the war, or something else. Rather than guess, I'd suggest checking the text of ""Estrangement"" directly; it's a short piece and the description should be easy to locate.",


### ❓ Reflect — Hallucination
1. Which model hallucinated more? Was it confident when wrong?
2. What mitigation (RAG, temperature=0, structured output) would help most for your specific prompts?
3. 'The model made it up' is not an explanation. What IS the architectural reason hallucination occurs?

1. GPT-4o hallucinated for all three prompts, albeit but only in the rare facts category. Claude Fable did not hallucinate for two of the three but partially hallucinated for the first and second prompts, shockingly getting most of both right, thereby proving AI really is eating most all human knowledge. All failed to hallucinate when provided prompts confidently and with great detail describing entirely made-up events.

2. To avoid hallucinating in the area of rare facts, I would suggest temperature=0 but also a larger model e.g. Fable and specific instructions/training permiting impoliteness or refusal to answer absent concrete knowledge as required to ensure accuracy.

3. These hallucinations occurred because the model generated each word in its output based on probabilities, which it computed after it had seen billions and billions of example sentences on the Internet. Probability is not however ground truth, and what may seem like almost certainly the correct answer probablistically may actually not be for exceedingly rare tail facts and sentences that the model may have never seen before, even on the vast Internet, such as never-before or rarely digitized texts and artistic works. In this lab, I fooled both GPT-4o and Claude with passages from rare books, as well as a likely never digitized symphonic score. Claude did better, avoiding answering in some places, but ultimately hallucinated with fine details, e.g. Oiwa's ghost committed her revenge strangulation with a strip of textile, not her own hair, and La Pasion Segun Marcos opens with tom-toms, not the harp, as Claude had outputted.

Reflection: The Model Selection Decision

1. GPT-4o summaries may easily receive higher ROUGE-1 scores than T5 despite being subjectively better because T5 may produce summaries with greater reference-text overlap. One evaluation metric that captures what ROUGE-1 misses would be a score by an LLM acting as a stylistic judge, e.g. rewarding synonyms superior in form and function more highly than interchangeable/identical tokens.

2. Given non-negotiable privacy and stringent cost constraints, I recommend Llama-3 for the following reasons:

2.1. First, it is open-source, which is paramount for privacy because you can host and run it locally on for example a maxed-out air-gapped Mac Studio that's leasable at a monthly fee just squeezing in under $500. This way no confidential data ever touches third-party servers.

2.2. Second, it is open-source, which I repeat for emphasis given a stringent cost constraint: The whole rig  would cost no more monthly to run than the extra electricity it would draw, e.g. $30-100. That's a good thing given most of the already quite tight budget would go to leasing otherwise quite expensive hardware.

2.3. There are trade-offs, which are thankfully at least partially mitigated. It's no Claude quality-wise, comparitively lacking in summary detail and more prone to hallucination. But that said, Llama-3's not far off from OpenAI's proprietary GPT-4o in either category. While it lost out to GPT-4o in terms of latency, it just managed to beat out Claude, at least in our experiments.

3. I would modify the prompt to state the following:

3.1. Key-claims must contain exactly three items;

3.2. If the article does not contain a third claim, state facts from the article as the third claim;

3.3 DO NOT invent information;

3.4 Do not provide a fourth claim;

3.5 Check your response prior to submission: if no third claim, then add a third claim; if fourth claim, then delete fourth claim.

3.6. For the code, I would require a schema-based output that the model would have to match, i.e.: key_claim 1; key_claim 2; key_claim 3.

4.1 Hallucinations happen because models generate each word in an output based on probabilities, which a model computes after it has seen billions and billions of example sentences on the Internet.

4.2. Probability is not however ground truth, and what may seem like almost certainly the correct answer probablistically may actually not be for exceedingly rare facts and sentences that the model may have never seen before, even on the vast Internet, such as rarely digitized texts and artistic works.

4.3. While bigger models hallucinate less often, as they've seen more, have better reasoning, and may have also been specifically trained to avoid common hallucination traps, or conversely to simply if impolitely refuse to answer absent concrete information, imagine an illustrative albeit borderline fantastical scenario of asking the biggest and best model to tell you specific details about a recently discovered ancient Egyptian inscription that has never been photographed let alone digitized; the model cannot actually know what the inscription says lacking eyes and a plane ticket to Egypt and can only either refuse to answer or provide the most probably right answer that, while convincing, is entirely fabricated as it is based on adjacent documents in Egyptology that it has seen before.